# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hassaan-Raza/FlyRank-Intership/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

One row = one content item (content_hash_id), with metrics aggregated across
a full calendar month. Time window: March 2026 (2026-03-01 to 2026-03-31),
a mid-panel month, not the sealed final month (2026-06) reserved for testing.

In [1]:
import duckdb
from google.colab import userdata

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")
REL = 'hf://datasets/FlyRank/internship-warehouse'

grain_check = con.sql(f"""
    SELECT content_hash_id, COUNT(*) as row_count
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY content_hash_id
    LIMIT 10
""").df()
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  row_count
0  content_b7e512995f79d5a6         31
1  content_05597932fe4da067         31
2  content_905aa32a0230694e         31
3  content_05434271b257bb68         31
4  content_d056587ff7faca0c         31
5  content_bfd1e41c2af250c8         31
6  content_2662845f598544ef         31
7  content_22610b0934f8825e         31
8  content_712c365258cee05c         31
9  content_476c37c366920c1b         31


## 2. Fields: feature / label / context / excluded

Features: impressions, clicks, avg_position, sessions, days_with_impressions
(observable signals, safe to use).

Label/proxy: a decline flag built from impressions dropping between the
first and second half of the window.

Context: content_hash_id, client_hash_id (for joins and grouping only,
not features themselves).

Excluded: any FlyRank product-computed score (health_score, priority_score,
action_type), and any row where ga4_data_available is FALSE. Excluded because
product scores would let the model copy an existing rule instead of finding
real signal, and rows without tracking yet would misrepresent "no traffic" as
real zero traffic.

In [2]:
# Query 1: row count and date span for this slice
span_check = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           MIN(report_date) as earliest_date,
           MAX(report_date) as latest_date
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(span_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows earliest_date latest_date
0     9841378    2026-03-01  2026-03-31


In [3]:
# Query 2: availability check, filtered with IS TRUE
availability_check = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) as available_rows
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(availability_check)
print(f"Survival rate: {availability_check['available_rows'][0] / availability_check['total_rows'][0]:.1%}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  available_rows
0     9841378          413966
Survival rate: 4.2%


In [6]:
missing_check = con.sql(f"""
    SELECT
        COUNT(*) as total_rows,
        COUNT(*) - COUNT(gsc_impressions) as missing_impressions,
        COUNT(*) - COUNT(gsc_avg_position) as missing_position
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(missing_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  missing_impressions  missing_position
0     9841378                    0           6230317


In [7]:
features = con.sql(f"""
    SELECT
        content_hash_id,
        SUM(gsc_impressions) as total_impressions,
        SUM(gsc_clicks) as total_clicks,
        AVG(gsc_avg_position) as avg_position,
        SUM(ga4_sessions) as total_sessions,
        COUNT(DISTINCT report_date) FILTER (WHERE gsc_impressions > 0) as days_with_impressions
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE ga4_data_available IS TRUE
    GROUP BY content_hash_id
""").df()
print(features.head(10))
print(f"\nShape: {features.shape}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  total_impressions  total_clicks  avg_position  \
0  content_810cf06597918291              257.0           1.0     11.186123   
1  content_b813c73d7000b3b1              180.0           1.0      8.674734   
2  content_5a77dbf5671c5a65            19657.0         199.0      4.532382   
3  content_f5e11209b398d173               47.0           0.0     10.045000   
4  content_8f3fa2db89105948                1.0           0.0      7.000000   
5  content_278030b007943b07              319.0           7.0      5.914484   
6  content_237e63fc00c8c7ad             6106.0          39.0      7.108246   
7  content_347fbafb77d3ae37              396.0           4.0     21.671343   
8  content_15bd72d24e0a0b08              428.0           2.0     13.669138   
9  content_6f4cc70af7be346e              133.0           5.0      9.234436   

   total_sessions  days_with_impressions  
0            42.0                     19  
1             7.0                      7  
2           

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Data limits

This slice only covers March 2026 for clients whose GA4 tracking had already
started by that point. Only 4.2% of rows have ga4_data_available IS TRUE
(413,966 of 9,841,378), so any GA4-dependent feature (sessions, engagement)
is only usable on a small fraction of this month's data, worth treating as
a hard constraint, not a detail. GSC-only rows (no GA4 yet) can still
misrepresent "no traffic" as real zero traffic if this filter is skipped.
This window also can't tell us anything about seasonality beyond one month,
or about clients who joined after March.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.